# Part 2 — Stochastic Supply Chain MILP: EV vs PI vs SP, and Progressive Hedging

Part 1 was deterministic — the model knew the future exactly. Rolling horizon was a crude
way to limit foresight: re-optimise when surprised. **This notebook does it properly.**

Two things happen here:

1. **§4 — the three-strategy comparison.** The same network, solved three ways, then each
   strategy's *here-and-now* decision is stress-tested across every scenario. This is where
   the case for stochastic programming is either made or not.
   - **EV** (Expected Value) — solve the mean-demand problem, commit to its answer
   - **PI** (Perfect Information / wait-and-see) — clairvoyant, a lower bound
   - **SP** (Stochastic Program) — hedge across the distribution, respecting nonanticipativity
2. **§6–7 — Progressive Hedging**, the decomposition algorithm that makes SP affordable at
   real scale, plus the block-asynchronous variant (Eckstein, Watson & Woodruff 2025).


**Network.** Mining → Processing → Manufacturing, two sites per tier, serving two demand
regions. Intra-region transport is cheap, cross-region is 4× dearer. Region 2 grows much
faster than Region 1, so the optimal answer shifts the network's centre of gravity over
time. Legacy assets retire on a **staggered** schedule (years 7/10/12/15/16/19), forcing
replacement waves throughout the horizon rather than one cliff at the end.

Facility siting is the binary part: `y[s,v,k]` = build unit *k* at site *s* decided in
year *v*, with a per-site construction lead time.

**Uncertainty:** Region 2's demand growth rate — 1% / 7% / 14% with probabilities
0.30 / 0.40 / 0.30. Region 1 is known.

**Runtime:** ~4–7 minutes.

## Legend / Notation

### Cost units in this instance
All costs are arbitrary money units, calibrated so the tradeoffs bite:

| Item | Magnitude |
|---|---|
| Build one facility unit (`capex0`) | 1,800 – 5,000 |
| Capacity per unit built (`cap_unit`) | 90 – 110 throughput units/yr |
| Variable cost, one delivered unit (all 3 tiers, ~53% chain yield) | ~10 – 13 |
| Transport per unit | 0.4 intra-region, 1.6 cross-region |
| Unmet-demand penalty (`slack_pen`) | 45 default; swept 30 – 250 in Part 2 |

### Time and horizon
| Symbol | Meaning |
|---|---|
| `T` | horizon length (years) |
| `t` | operating year |
| `v` | **vintage** — the year an asset's build was decided |
| `r` | discount rate (5% here) |
| $\delta_t = 1/(1+r)^t$ | discount factor |
| `invest_years` / `iy` | years in which a build decision may be taken |
| `L` (`d.life`) | asset life (20 yr) |
| `lead` | construction lead time (2–3 yr by site) |

### Capital cost
| Symbol | Meaning |
|---|---|
| `CRF` | capital recovery factor, $\dfrac{r(1+r)^L}{(1+r)^L-1}$ — converts a lump sum into an equivalent annuity |
| `lumpsum` | charge the full capital cost at the decision year |
| `annualized` | charge `CRF` × cost in each operating year inside the horizon |

### Efficiency
| Symbol | Meaning |
|---|---|
| $\eta(v,t)$ | yield of a vintage-$v$ asset operating in year $t$ |
| $\bar\eta$ | ceiling (thermodynamic / technical limit) |
| $\alpha$ | frontier improvement rate — applies to **new builds** |
| $\beta$ | within-life improvement rate for an **existing** asset, $\beta < \alpha$ |
| $\bar\Delta$ | cap on total lifetime retrofit gain for one asset |

### Learning
| Symbol | Meaning |
|---|---|
| `LR` | learning rate — fractional cost drop per **doubling** of cumulative capacity |
| $b = -\log_2(1-LR)$ | Wright's law exponent |
| $Q$, $Q_0$ | cumulative capacity; incumbent base |
| $C(Q)$ | **cumulative** capex — this is what gets linearized, not unit cost |
| $\lambda$ | SOS2 interpolation weights on the piecewise curve |

### Rolling horizon
| Symbol | Meaning |
|---|---|
| `W` | foresight window — how far ahead each solve can see |
| `delta` ($\Delta$) | roll step — years **committed** before re-solving |
| `decision_zone` (`dz`) | years inside the window where binary builds are allowed |
| `tail_continuous` | beyond `dz`, capacity is continuous (recommended) rather than banned |

### Stochastic programming — the three cases

| Case | What it knows when choosing year 1 | Symbol |
|---|---|---|
| **PI** — Perfect Information | everything; solves each future separately | **WS** (wait-and-see) |
| **SP** — Stochastic Program | the probability distribution; hedges across all futures | **RP** (recourse problem) |
| **EV** — Expected Value | only the mean; a point forecast | **EEV** (expected result of the EV solution) |

| Quantity | Definition | Reading |
|---|---|---|
| **WS** | solve each scenario deterministically, probability-weight the objectives | lower bound; **not achievable** |
| **RP** | stochastic optimum, respecting nonanticipativity | what you'd actually do |
| **EEV** | fix the EV year-1 decision, re-optimise stage 2 in **every** scenario, weight | what a point forecast costs you |
| **EVPI** | $\text{RP}-\text{WS}$ | value of foresight — **cannot be bought** |
| **VSS** | $\text{EEV}-\text{RP}$ | value of modelling uncertainty — **under your control** |

Guaranteed: $\;\text{WS} \le \text{RP} \le \text{EEV}$.

**All three must be *evaluated expectations* from the same machinery.** Reading RP off
`ef.ObjVal` (a gap-terminated solve value) while EEV comes from the evaluation path mixes two
different measurements and can produce a negative VSS, which is impossible. See §4.

| Symbol | Meaning |
|---|---|
| `pen` (`slack_pen`) | penalty per unit of **unmet demand** — the recourse cost of being caught short. Raising it makes *under*-building painful while over-building stays merely wasteful; that asymmetry is what makes VSS nonzero. |
| `EV_builds` / `SP_builds` | year-1 units each strategy commits. **The diagnostic:** VSS is nonzero only when these differ. |
| $\pi_i$ | probability of scenario $i$ |
| `stage1_years` (`s1`) | years whose decisions must be **nonanticipative** (identical across scenarios) |
| $\mathcal{N}$ | the nonanticipativity subspace |

### Progressive Hedging
| Symbol | Meaning |
|---|---|
| $\rho$ | penalty parameter on deviation from the average — the critical tuning knob |
| $z$ | probability-weighted average of scenario solutions (projection onto $\mathcal{N}$) |
| $w_i$ | Lagrange multipliers for scenario $i$ |
| `block_frac` | fraction of scenarios re-solved per iteration (1.0 = classic synchronous PH; < 1 = APH-style block asynchronous) |
| residual | $\sqrt{\sum_i \pi_i \lVert x_i - z\rVert^2}$ — how far scenarios still disagree |

## 1. Setup

The `pip` build of Gurobi ships a **restricted licence capped near 2,000 variables and
2,000 constraints**. Every model here stays under it. If you scale `T` or add sites you
will hit the cap — swap in your university's WLS licence:

```python
env = gp.Env(params={"WLSACCESSID": "<your WLS access id>", "WLSSECRET": "<your WLS secret>", "LICENSEID": <your licence id>})
```
and pass `env=env` into `gp.Model()` inside `build()`.

> **Do not paste a licence key into a notebook.** A key committed to a repository is exposed the moment the repository is shared, and deleting it in a later commit does not remove it from history — the only fix is to rotate the key. Read it from an environment variable or a Colab secret instead; `Part4c_exact_MIQP.ipynb` cell 2 shows the pattern.


In [ ]:
!pip install gurobipy --quiet
import math, time, random, itertools
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
print("gurobipy", gp.gurobi.version())

## 2. The model

Identical to Part 1 (see that notebook for the derivation of vintage efficiency, CRF
annualization and the SOS2 learning curve). Reproduced here so this notebook stands alone.
The only addition is a `demand` override so each scenario gets its own demand path, and an
`into`/`prefix` hook so scenario blocks can be stacked into one shared model for the
extensive form.

In [ ]:
class Data:
    def __init__(self, T=20, r=0.05):
        self.T = T
        self.r = r
        self.years = list(range(1, T + 1))
        self.df = {t: 1.0 / (1 + r) ** t for t in self.years}

        self.regions = ['R1', 'R2']
        self.mines = ['M1', 'M2']
        self.procs = ['P1', 'P2']
        self.fabs  = ['F1', 'F2']
        self.sites = self.mines + self.procs + self.fabs
        # home region of each site
        self.home = {'M1':'R1','M2':'R2','P1':'R1','P2':'R2','F1':'R1','F2':'R2'}

        self.cap_unit = {'M1':110,'M2':110,'P1':100,'P2':100,'F1':90,'F2':90}
        self.max_builds = 3
        self.life = 20            # asset life (yrs)
        self.lead = {'M1':2,'M2':2,'P1':3,'P2':3,'F1':2,'F2':2}

        # capex per build (year-0 cost, before learning)
        self.capex0 = {'M1':1800,'M2':2000,'P1':3300,'P2':3100,'F1':2900,'F2':2700}
        # variable opex per unit throughput
        self.opex = {'M1':1.2,'M2':1.4,'P1':2.0,'P2':2.2,'F1':2.5,'F2':2.3}

        # transport cost per unit: cheap intra-region, dear cross-region
        self.tc = lambda a, b: 0.4 if self.home[a] == self.home[b] else 1.6
        self.tc_dem = lambda f, rg: 0.4 if self.home[f] == rg else 1.6

        # ---- efficiency (yield) : vintage frontier + within-life improvement ----
        self.eta_bar = {'P':0.95, 'F':0.93}   # ceiling
        self.eta_0   = {'P':0.80, 'F':0.78}   # vintage-1 frontier
        self.alpha   = {'P':0.030,'F':0.025}  # frontier improvement / yr
        self.beta    = {'P':0.010,'F':0.008}  # within-life improvement / yr
        self.dbar    = {'P':0.05, 'F':0.05}   # max lifetime gain for one asset

        # mining yield is constant (ore -> concentrate)
        self.eta_mine = 0.90
        self.eta_min = 0.60   # clamp: legacy assets can't be arbitrarily bad

        # brownfield: legacy units already operating in year 1 (vintage <= 0)
        # legacy: (units, vintage, retirement year). Staggered retirements create
        # replacement waves spread across the horizon, not one cliff at the end.
        self.legacy = {'M1':(2,-6, 7), 'M2':(2,-6, 10),
                       'P1':(2,-3,12), 'P2':(2,-3, 15),
                       'F1':(2,-1,16), 'F2':(2,-1, 19)}

        # ---- demand: R2 grows much faster than R1 ----
        self.D = {}
        for t in self.years:
            self.D['R1', t] = 110 * (1.025 ** (t - 1))
            self.D['R2', t] = 85 * (1.055 ** (t - 1))
        self.slack_pen = 45.0     # unmet demand penalty per unit

        # ---- learning ----
        self.learn_frac = 0.70    # share of capex that is learnable technology
        self.learn_sites = self.procs + self.fabs   # recovery tech learns; mining is mature
        self.LR = 0.20            # 15% cost drop per doubling of cumulative capacity
        self.Q0 = 380.0           # incumbent cumulative capacity
        self.c_floor_frac = 0.55  # floor as fraction of c0
        self.g_exog = 0.035       # exogenous capex decline / yr (for 'exogenous' mode)

    def eta(self, tier, v, t):
        """Yield of a tier-`tier` asset of vintage v, operating in year t."""
        eb, e0 = self.eta_bar[tier], self.eta_0[tier]
        a, b, db = self.alpha[tier], self.beta[tier], self.dbar[tier]
        e_new = eb - (eb - e0) * (1 - a) ** (v - 1)
        e_new = max(e_new, self.eta_min)
        e_t   = eb - (eb - e_new) * (1 - b) ** (t - v)
        return max(self.eta_min, min(e_new + db, e_t))

    def crf(self):
        r, L = self.r, self.life
        return r * (1 + r) ** L / ((1 + r) ** L - 1)

    def tier_of(self, s):
        return 'M' if s in self.mines else ('P' if s in self.procs else 'F')

In [ ]:
# ---------------- CAPEX PV MULTIPLIERS ----------------
def capex_pv_multiplier(d, s, dec_year, mode, y_start=1, y_end=None):
    """PV (discounted to yr 0) of $1 of capex for a facility decided in `dec_year`.

    annualized : CRF x $1 charged each operating yr inside [y_start, y_end]
    lumpsum    : full $1 paid at the decision year
    """
    y_end = y_end or d.T
    online = dec_year + d.lead[s]
    if mode == 'lumpsum':
        return d.df.get(dec_year, 0.0)
    last = min(online + d.life - 1, y_end)
    if last < online:
        return 0.0
    return d.crf() * sum(d.df[t] for t in range(online, last + 1) if t in d.df)

In [ ]:
# ---------------- LEARNING BREAKPOINTS ----------------
def learning_breakpoints(d, qmax, nbp=7):
    """Breakpoints for cumulative capex C(Q) under Wright's law with a floor."""
    b = -math.log2(1 - d.LR)
    c0 = 1.0                                   # normalised unit cost multiplier
    cf = d.c_floor_frac

    def unit(q):
        return max(cf, c0 * (q / d.Q0) ** (-b))

    def cum(q):
        # integral of unit() from Q0 to q, done numerically (robust w/ the floor)
        n, lo, acc = 400, d.Q0, 0.0
        if q <= lo:
            return 0.0
        h = (q - lo) / n
        for i in range(n):
            acc += 0.5 * (unit(lo + i * h) + unit(lo + (i + 1) * h)) * h
        return acc

    Qbp = [d.Q0 + (qmax - d.Q0) * (i / (nbp - 1)) for i in range(nbp)]
    Cbp = [cum(q) for q in Qbp]
    return Qbp, Cbp, unit

In [ ]:
# ---------------- MODEL BUILDER ----------------
def build(d, invest_years=None, capex_mode='annualized', learning='none',
          y_start=1, y_end=None, fixed_builds=None, forced_zero_after=None,
          relax_int_after=None, quiet=True, demand=None, into=None, prefix='',
          mipgap=None):
    """
    invest_years      : years in which a build decision may be taken
    capex_mode        : 'annualized' | 'lumpsum'
    learning          : 'none' | 'exogenous' | 'endogenous'
    y_start,y_end     : operating window (for rolling horizon)
    fixed_builds      : {(site, dec_year): 0/1} decisions frozen from earlier rolls
    forced_zero_after : no new builds decided after this year
    relax_int_after   : builds decided after this year are continuous in [0,1]
    """
    y_end = y_end or d.T
    yrs = [t for t in d.years if y_start <= t <= y_end]
    if invest_years is None:
        invest_years = list(d.years)
    IY = [v for v in invest_years if y_start <= v <= y_end]
    if forced_zero_after is not None:
        IY = [v for v in IY if v <= forced_zero_after]

    def usable(s, v):
        # a decision is only meaningful if the asset comes online inside the window
        return v + d.lead[s] <= y_end
    site_IY = {s: [v for v in IY if usable(s, v)] for s in d.sites}

    D = demand if demand is not None else d.D
    if into is not None:
        m = into
    else:
        m = gp.Model()
        if quiet:
            m.Params.OutputFlag = 0
        m.Params.MIPGap = 0.005 if mipgap is None else mipgap

    # --- build decisions: y[s,v,k] = k-th unit at site s decided in year v ---
    idx = [(s, v, k) for s in d.sites for v in site_IY[s] for k in range(d.max_builds)]
    y = {}
    for (s, v, k) in idx:
        cont = (relax_int_after is not None and v > relax_int_after)
        y[s, v, k] = m.addVar(vtype=GRB.CONTINUOUS if cont else GRB.BINARY,
                              ub=1.0, name=f"{prefix}y_{s}_{v}_{k}")
    # symmetry breaking within a site-year
    for s in d.sites:
        for v in site_IY[s]:
            for k in range(d.max_builds - 1):
                m.addConstr(y[s, v, k] >= y[s, v, k + 1])

    if fixed_builds:
        for (s, v), val in fixed_builds.items():
            for k in range(d.max_builds):
                if (s, v, k) in y:
                    pass
        # handled by caller via 'prebuilt' capacity instead

    # --- prebuilt capacity inherited from earlier rolls: {(site, online_yr, vintage): units}
    prebuilt = fixed_builds or {}

    def online_units(s, t):
        """Expression for number of units of site s available in year t, by vintage."""
        terms = []
        if s in d.legacy:
            ln, lv, lret = d.legacy[s]
            if t <= lret:
                terms.append((lv, float(ln)))
        for v in site_IY[s]:
            on = v + d.lead[s]
            if on <= t <= on + d.life - 1:
                terms.append((v, gp.quicksum(y[s, v, k] for k in range(d.max_builds))))
        for (ps, pv), n in prebuilt.items():
            if ps == s:
                on = pv + d.lead[s]
                if on <= t <= on + d.life - 1:
                    terms.append((pv, n))
        return terms

    # --- throughput, vintage indexed at P and F tiers ---
    thr = {}
    for s in d.procs + d.fabs:
        for t in yrs:
            for (v, _) in online_units(s, t):
                if (s, v, t) not in thr:
                    thr[s, v, t] = m.addVar(name=f"thr_{s}_{v}_{t}")
    ext = {(s, t): m.addVar(name=f"ext_{s}_{t}") for s in d.mines for t in yrs}

    # --- arc flows ---
    fmp = {(a, b, t): m.addVar() for a in d.mines for b in d.procs for t in yrs}
    fpf = {(a, b, t): m.addVar() for a in d.procs for b in d.fabs for t in yrs}
    ffr = {(a, g, t): m.addVar() for a in d.fabs for g in d.regions for t in yrs}
    slk = {(g, t): m.addVar() for g in d.regions for t in yrs}

    # --- capacity constraints ---
    for t in yrs:
        for s in d.mines:
            cap = gp.quicksum(n * d.cap_unit[s] for (_, n) in online_units(s, t))
            m.addConstr(ext[s, t] <= cap)
        for s in d.procs + d.fabs:
            for (v, n) in online_units(s, t):
                m.addConstr(thr[s, v, t] <= n * d.cap_unit[s])

    # --- flow balances ---
    for t in yrs:
        for s in d.mines:
            m.addConstr(d.eta_mine * ext[s, t] == gp.quicksum(fmp[s, b, t] for b in d.procs))
        for s in d.procs:
            vints = [v for (v, _) in online_units(s, t)]
            m.addConstr(gp.quicksum(fmp[a, s, t] for a in d.mines)
                        == gp.quicksum(thr[s, v, t] for v in vints))
            m.addConstr(gp.quicksum(d.eta('P', v, t) * thr[s, v, t] for v in vints)
                        == gp.quicksum(fpf[s, b, t] for b in d.fabs))
        for s in d.fabs:
            vints = [v for (v, _) in online_units(s, t)]
            m.addConstr(gp.quicksum(fpf[a, s, t] for a in d.procs)
                        == gp.quicksum(thr[s, v, t] for v in vints))
            m.addConstr(gp.quicksum(d.eta('F', v, t) * thr[s, v, t] for v in vints)
                        == gp.quicksum(ffr[s, g, t] for g in d.regions))
        for g in d.regions:
            m.addConstr(gp.quicksum(ffr[f, g, t] for f in d.fabs) + slk[g, t] >= D[g, t])

    # --- capex term ---
    # capex(s,v) = site adder (never learns) + technology cost (may learn)
    LS = set(d.learn_sites)
    adder = {s: d.capex0[s] * (1 - d.learn_frac) if s in LS else d.capex0[s]
             for s in d.sites}
    tech_rate = (sum(d.capex0[s] * d.learn_frac / d.cap_unit[s] for s in LS)
                 / len(LS))                       # $ per unit of capacity

    capex_expr = gp.LinExpr()
    for s in d.sites:                              # site adders, all modes
        for v in site_IY[s]:
            mult = capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
            for k in range(d.max_builds):
                capex_expr += mult * adder[s] * y[s, v, k]

    if learning in ('none', 'exogenous'):
        for s in LS:
            for v in site_IY[s]:
                mult = capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
                decay = (1 - d.g_exog) ** (v - 1) if learning == 'exogenous' else 1.0
                rate = tech_rate * max(decay, d.c_floor_frac)
                for k in range(d.max_builds):
                    capex_expr += mult * rate * d.cap_unit[s] * y[s, v, k]
    else:  # endogenous: SOS2 on CUMULATIVE technology cost
        qmax = d.Q0 + sum(d.cap_unit[s] * d.max_builds for s in LS) * max(
            1, len(IY) // 3)
        Qbp, Cbp, _ = learning_breakpoints(d, qmax)
        allv = sorted(set(v for s in LS for v in site_IY[s]))
        prevC = None
        for v in allv:
            Qv = m.addVar(lb=d.Q0, ub=qmax, name=f"Qcum_{v}")
            Cv = m.addVar(lb=0, name=f"Ccum_{v}")
            lam = [m.addVar(lb=0, ub=1, name=f"lam_{v}_{j}") for j in range(len(Qbp))]
            m.addConstr(gp.quicksum(lam) == 1)
            m.addConstr(Qv == gp.quicksum(l * q for l, q in zip(lam, Qbp)))
            m.addConstr(Cv == gp.quicksum(l * c for l, c in zip(lam, Cbp)))
            m.addSOS(GRB.SOS_TYPE2, lam)           # <-- the essential restriction
            pre_q = sum(d.cap_unit[ps] * n for (ps, pv), n in prebuilt.items()
                        if ps in LS and pv <= v)
            m.addConstr(Qv == d.Q0 + pre_q + gp.quicksum(
                d.cap_unit[s] * y[s, vv, k]
                for s in LS for vv in site_IY[s] if vv <= v
                for k in range(d.max_builds)))
            cand = [capex_pv_multiplier(d, s, v, capex_mode, y_start, y_end)
                    for s in LS if v in site_IY[s]]
            mult = sum(cand) / len(cand) if cand else 0.0
            capex_expr += mult * tech_rate * (Cv - (prevC if prevC is not None else 0))
            prevC = Cv

    # --- operating + transport + penalty ---
    op = gp.LinExpr()
    for t in yrs:
        w = d.df[t]
        for s in d.mines:
            op += w * d.opex[s] * ext[s, t]
        for s in d.procs + d.fabs:
            for (v, _) in online_units(s, t):
                op += w * d.opex[s] * thr[s, v, t]
        for a in d.mines:
            for b in d.procs:
                op += w * d.tc(a, b) * fmp[a, b, t]
        for a in d.procs:
            for b in d.fabs:
                op += w * d.tc(a, b) * fpf[a, b, t]
        for f in d.fabs:
            for g in d.regions:
                op += w * d.tc_dem(f, g) * ffr[f, g, t]
        for g in d.regions:
            op += w * d.slack_pen * slk[g, t]

    if into is None:
        m.setObjective(capex_expr + op, GRB.MINIMIZE)
    m._y, m._siteIY, m._IY, m._slk, m._ffr, m._d = y, site_IY, IY, slk, ffr, d
    m._capex_expr, m._op_expr = capex_expr, op
    m._adder, m._tech_rate = adder, tech_rate
    if into is not None:
        return m, y, capex_expr + op, slk
    return m


def build_plan(m):
    """Extract {(site, decision_year): n_units} from a solved model."""
    out = {}
    for (s, v, k), var in m._y.items():
        if var.X > 0.5:
            out[s, v] = out.get((s, v), 0) + 1
    return out

## 3. Scenario tree and nonanticipativity

Define a tree with stages $t\in 1..T$ and leaf scenarios $i\in 1..n$ with probabilities
$\pi_i$. A plan is **implementable** only if scenarios that are indistinguishable at stage
$t$ take the *same* stage-$t$ decision:

$$\mathcal{N} = \{z \;|\; z_{it}=z_{jt} \text{ whenever } i,j\in S(N),\, N\in\mathcal{T}_t\}$$

The stochastic program is
$$\min_{x}\ \sum_i \pi_i h_i(x_i) \quad\text{s.t.}\quad Mx \in \mathcal{N}$$

**Nonanticipativity is the rigorous version of "limited foresight."** Where Part 1's rolling
horizon truncated at an ad hoc $W$ and re-optimised when surprised, this optimises *against
a distribution* of futures — so the here-and-now decision is optimal in expectation over
the whole tree.

Here it's a two-stage problem: **stage 1** = year-1 build decisions (must be identical
across scenarios), **stage 2** = all later builds and all operations (may differ). Solving
this directly — the **extensive form** — replicates the entire model once per scenario,
which is exactly why it becomes intractable and why §6 exists.

In [ ]:
def scenarios(d, growths=((0.010, 0.30), (0.070, 0.40), (0.140, 0.30))):
    """Demand scenarios: uncertainty is Region-2 growth. Returns list of (name, prob, D)."""
    out = []
    for j, (g, p) in enumerate(growths):
        D = {}
        for t in d.years:
            D['R1', t] = d.D['R1', t]                 # R1 known
            D['R2', t] = 105 * ((1 + g) ** (t - 1))    # R2 uncertain
        out.append((f"s{j}_g{g:.3f}", p, D))
    return out


def extensive_form(d, scens, invest_years, stage1_years, quiet=True, mipgap=None):
    """One monolithic MILP over all scenarios, with explicit nonanticipativity."""
    m = gp.Model()
    if quiet:
        m.Params.OutputFlag = 0
    m.Params.MIPGap = 0.005 if mipgap is None else mipgap
    ys, objs = [], gp.LinExpr()
    for (nm, p, D) in scens:
        _, y, obj, _ = build(d, invest_years=invest_years, demand=D,
                             into=m, prefix=nm + "_", mipgap=mipgap)
        ys.append(y)
        objs += p * obj
    # nonanticipativity: stage-1 builds identical across scenarios
    for key in ys[0]:
        if key[1] in stage1_years:
            for j in range(1, len(ys)):
                m.addConstr(ys[0][key] == ys[j][key], name=f"NA_{key}_{j}")
    m.setObjective(objs, GRB.MINIMIZE)
    m._ys = ys
    return m


def stage1_keys(y, stage1_years):
    return [k for k in y if k[1] in stage1_years]


def subproblem(d, D, invest_years, stage1_years):
    """One scenario subproblem, kept open so PH can re-set its objective each iteration."""
    m = build(d, invest_years=invest_years, demand=D)
    m._base_obj = m._capex_expr + m._op_expr
    m._s1 = stage1_keys(m._y, stage1_years)
    return m


def ph(d, scens, invest_years, stage1_years, rho=None, iters=40, tol=1e-4,  # noqa
       block_frac=1.0, seed=0, verbose=False):
    """
    Progressive hedging. block_frac < 1 gives the APH-style block-asynchronous
    variant: only a subset of subproblems is re-solved each iteration.

    Binary stage-1 trick: for x in {0,1}, x^2 = x, so
        (rho/2)||x - z||^2  ==  (rho/2) * [ x(1 - 2z) + z^2 ]
    which is LINEAR. Subproblems stay MILPs -- no MIQP needed.
    """
    rng = random.Random(seed)
    subs = [(nm, p, subproblem(d, D, invest_years, stage1_years))
            for (nm, p, D) in scens]
    n = len(subs)
    if rho is None:                      # scale rho to the capex of one build
        rho = 0.5 * sum(d.capex0.values()) / len(d.capex0)

    keys = subs[0][2]._s1
    w = [{k: 0.0 for k in keys} for _ in range(n)]
    xv = [{k: 0.0 for k in keys} for _ in range(n)]
    z = {k: 0.0 for k in keys}
    hist = []
    solved_count = 0

    nblock = max(1, int(round(block_frac * n)))
    cursor = 0
    for it in range(iters):
        # Fairness (Assumption A3 in Eckstein et al.): every scenario must be
        # re-solved at least once every M iterations. A deterministic round-robin
        # guarantees this; pure random sampling does not.
        if it == 0:
            I = list(range(n))                     # everyone once, to seed x_i
        elif nblock >= n:
            I = list(range(n))
        else:
            I = [(cursor + j) % n for j in range(nblock)]
            cursor = (cursor + nblock) % n
        for i in I:
            nm, p, m = subs[i]
            obj = gp.LinExpr(m._base_obj)
            for k in keys:
                # linear Lagrange term + linearised quadratic penalty
                obj += w[i][k] * m._y[k]
                obj += 0.5 * rho * (m._y[k] * (1 - 2 * z[k]) + z[k] ** 2)
            m.setObjective(obj, GRB.MINIMIZE)
            m.optimize()
            solved_count += 1
            for k in keys:
                xv[i][k] = m._y[k].X
        # average (projection onto the nonanticipativity subspace)
        z = {k: sum(subs[i][1] * xv[i][k] for i in range(n)) for k in keys}
        # multiplier update
        for i in range(n):
            for k in keys:
                w[i][k] += rho * (xv[i][k] - z[k])
        resid = math.sqrt(sum(subs[i][1] * (xv[i][k] - z[k]) ** 2
                              for i in range(n) for k in keys))
        hist.append(resid)
        if verbose:
            print(f"  it {it:3d} block={len(I)}/{n} resid={resid:.5f}")
        if resid < tol and it > 0:
            break
    return dict(z=z, resid=hist, iters=len(hist), subsolves=solved_count,
                rho=rho, x=xv)


def evaluate_stage1(d, scens, invest_years, stage1_years, z, quiet=True):
    """Fix stage-1 builds to a rounded z, re-optimise stage 2 in every scenario."""
    fixed = {k: (1 if z.get(k, 0) > 0.5 else 0) for k in z}
    tot = 0.0
    for (nm, p, D) in scens:
        m = build(d, invest_years=invest_years, demand=D)
        for k, val in fixed.items():
            if k in m._y:
                m._y[k].LB = m._y[k].UB = val
        m.optimize()
        if m.SolCount == 0:
            return None
        tot += p * m.ObjVal
    return tot


def wait_and_see(d, scens, invest_years):
    """Each scenario solved with full knowledge -> lower bound; gap to RP is EVPI."""
    tot = 0.0
    for (nm, p, D) in scens:
        m = build(d, invest_years=invest_years, demand=D)
        m.optimize()
        tot += p * m.ObjVal
    return tot


def mean_value_stage1(d, scens, invest_years, stage1_years, mipgap=None):
    """Solve the deterministic mean-demand problem; return its stage-1 decisions."""
    Dm = {}
    for key in scens[0][2]:
        Dm[key] = sum(p * D[key] for (_, p, D) in scens)
    m = build(d, invest_years=invest_years, demand=Dm, mipgap=mipgap)
    m.optimize()
    return {k: m._y[k].X for k in stage1_keys(m._y, stage1_years)}


# ---------- EV / PI / SP strategy comparison ----------
def eval_strategy_by_scenario(d, scens, invest_years, stage1_years, s1_fix, mipgap=None):
    """Fix a stage-1 strategy, re-optimise stage 2 in EVERY scenario separately.
    Returns per-scenario cost and unmet demand -- the distribution, not just the mean."""
    rows = []
    for (nm, p, D) in scens:
        m = build(d, invest_years=invest_years, demand=D, mipgap=mipgap)
        for k, val in s1_fix.items():
            if k in m._y:
                m._y[k].LB = m._y[k].UB = (1 if val > 0.5 else 0)
        m.optimize()
        if m.SolCount == 0:
            rows.append(dict(scenario=nm, prob=p, cost=None, unmet=None))
            continue
        rows.append(dict(scenario=nm, prob=p, cost=m.ObjVal,
                         unmet=sum(v.X for v in m._slk.values())))
    return rows


def strategy_stage1(d, scens, invest_years, stage1_years, which, rho=300, iters=60):
    """Stage-1 decision produced by each strategy.
       'EV' = mean-demand deterministic;  'SP' = stochastic extensive form."""
    if which == 'EV':
        return mean_value_stage1(d, scens, invest_years, stage1_years)
    if which == 'SP':
        ef = extensive_form(d, scens, invest_years, stage1_years)
        ef.optimize()
        return {k: ef._ys[0][k].X for k in stage1_keys(ef._ys[0], stage1_years)}
    if which == 'PH':
        r = ph(d, scens, invest_years, stage1_years, rho=rho, iters=iters)
        return r['z']
    raise ValueError(which)


def perfect_info_by_scenario(d, scens, invest_years, mipgap=None):
    """PI / wait-and-see: each scenario optimised with full knowledge."""
    rows = []
    for (nm, p, D) in scens:
        m = build(d, invest_years=invest_years, demand=D, mipgap=mipgap)
        m.optimize()
        rows.append(dict(scenario=nm, prob=p, cost=m.ObjVal,
                         unmet=sum(v.X for v in m._slk.values())))
    return rows


def three_case_comparison(d, scens, invest_years, stage1_years, mipgap=1e-6):
    """EV / SP / PI computed through IDENTICAL evaluation machinery.

    PI  : each scenario solved deterministically, probability-weighted average (WS)
    SP  : stage-1 from the extensive form, FIXED, stage 2 re-optimised per scenario (RP)
    EV  : stage-1 from the mean-demand problem, FIXED, stage 2 re-optimised per scenario (EEV)

    Every number is an EVALUATED expectation. Nothing is a raw solve value.
    Theory: WS <= RP <= EEV, and per scenario PI <= SP and PI <= EV.
    """
    ef = extensive_form(d, scens, invest_years, stage1_years, mipgap=mipgap)
    ef.optimize()
    ef_obj = ef.ObjVal
    sp_fix = {k: ef._ys[0][k].X for k in stage1_keys(ef._ys[0], stage1_years)}
    ev_fix = mean_value_stage1(d, scens, invest_years, stage1_years, mipgap=mipgap)

    per = {}
    per['PI'] = perfect_info_by_scenario(d, scens, invest_years, mipgap=mipgap)
    per['SP'] = eval_strategy_by_scenario(d, scens, invest_years, stage1_years,
                                          sp_fix, mipgap=mipgap)
    per['EV'] = eval_strategy_by_scenario(d, scens, invest_years, stage1_years,
                                          ev_fix, mipgap=mipgap)
    exp = {k: sum(r['prob'] * r['cost'] for r in v) for k, v in per.items()}
    return dict(per=per, WS=exp['PI'], RP=exp['SP'], EEV=exp['EV'],
                ef_obj=ef_obj, sp_fix=sp_fix, ev_fix=ev_fix)


def scenarios_n(d, n, lo=0.01, hi=0.14, base=105.0):
    """Equiprobable discretization of the R2 growth rate into n scenarios.
    n=1 collapses to the mean (useful as a reference)."""
    if n == 1:
        gs = [0.5*(lo+hi)]
    else:
        gs = [lo + (hi-lo)*j/(n-1) for j in range(n)]
    p = 1.0/len(gs)
    out = []
    for j, g in enumerate(gs):
        D = {}
        for t in d.years:
            D['R1', t] = d.D['R1', t]
            D['R2', t] = base*((1+g)**(t-1))
        out.append((f's{j:02d}_g{g:.4f}", '.replace('", ', ''), p, D))
    return out


def ph_three_case(d, scens, invest_years, stage1_years, rho=300, iters=60,
                  block_frac=1.0, mipgap=1e-6):
    """EV / SP / PI at ANY scenario count -- no extensive form required.

    SP stage-1 comes from Progressive Hedging instead of the extensive form, so this
    scales past the point where the monolithic model stops fitting in memory/licence.
    WS and EEV are per-scenario solves and were never size-constrained.
    """
    r = ph(d, scens, invest_years, stage1_years, rho=rho, iters=iters,
           block_frac=block_frac)
    sp_fix = {k: (1.0 if v > 0.5 else 0.0) for k, v in r['z'].items()}
    ev_fix = mean_value_stage1(d, scens, invest_years, stage1_years, mipgap=mipgap)
    per = {
        'PI': perfect_info_by_scenario(d, scens, invest_years, mipgap=mipgap),
        'SP': eval_strategy_by_scenario(d, scens, invest_years, stage1_years,
                                        sp_fix, mipgap=mipgap),
        'EV': eval_strategy_by_scenario(d, scens, invest_years, stage1_years,
                                        ev_fix, mipgap=mipgap),
    }
    exp = {k: sum(x['prob']*x['cost'] for x in v) for k, v in per.items()}
    return dict(per=per, WS=exp['PI'], RP=exp['SP'], EEV=exp['EV'],
                sp_fix=sp_fix, ev_fix=ev_fix, ph=r)

In [ ]:
d = Data(T=12)
d.slack_pen = 160          # asymmetric recourse: under-building is expensive
iy, s1 = [1, 5, 9], [1]    # investment periods; stage-1 = year 1
sc = scenarios(d)
for nm, p, D in sc:
    print(f"{nm}   prob={p:.2f}   R2 demand: yr1={D['R2',1]:.0f} -> yr12={D['R2',12]:.0f}")

In [ ]:
# ---- shared plot style (run once) ----
import matplotlib.pyplot as plt, numpy as np
CB = {'blue':'#2E6F9E','orange':'#D97A2B','green':'#3F8F5B',
      'red':'#C0392B','grey':'#7F8C8D','purple':'#7D5BA6'}
plt.rcParams.update({'font.size':12,'axes.titlesize':13,'figure.dpi':120,
                     'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False})

In [ ]:
# FIG -- the scenario tree
fig, ax = plt.subplots(figsize=(6.8, 4.2))
for (nm, p, D), c in zip(sc, [CB['blue'], CB['grey'], CB['red']]):
    ax.plot(d.years, [D['R2', t] for t in d.years], '-o', lw=2.4, ms=4, color=c,
            label=f"R2 {nm.split('_g')[1]}  p={p:.2f}")
ax.plot(d.years, [d.D['R1', t] for t in d.years], 'k--', lw=2, label='R1 (known)')
ax.set(xlabel='year', ylabel='demand (units)',
       title='Uncertainty: Region 2 growth rate')
ax.legend(fontsize=10); plt.tight_layout(); plt.show()


### What exactly are the scenarios?

**Not random draws, and not Monte Carlo.** Three hand-specified demand paths with assigned
probabilities — a deliberately coarse **discretization** of a growth-rate distribution:

| Scenario | R2 growth | Probability |
|---|---|---|
| low | 1% / yr | 0.30 |
| mid | 7% / yr | 0.40 |
| high | 14% / yr | 0.30 |

Three things to be clear about:

1. **Only Region 2 is uncertain.** Region 1's demand is known exactly. That keeps the
   uncertainty one-dimensional and legible.
2. **The uncertain quantity is a single parameter** — the growth *rate* — not per-year noise.
   Each scenario is a smooth exponential path, so within a scenario the future is perfectly
   known. Realistic alternatives (year-on-year shocks, correlated regional demand, price
   uncertainty) would need a genuine multistage tree.
3. **Three scenarios is tiny.** Real practice runs dozens to hundreds, usually Monte Carlo
   draws followed by scenario reduction. EVPI and VSS computed from three points are coarse
   estimates, not converged ones.

Why only three here? The extensive form replicates the whole model per scenario, so it grows
linearly in $n$ and hits the `pip` licence cap (~2,000 vars) at around $n=5$. **That
constraint is exactly the motivation for §6** — PH solves scenarios separately, so it has no
such cap.


In [ ]:
one = build(d, invest_years=iy); one.update()
print(f"one deterministic scenario : {one.NumVars:5d} vars")
for n in [3, 5, 10, 50, 200]:
    print(f"  extensive form, n={n:3d}   : {one.NumVars*n:6d} vars"
          f"{'   <-- over the pip licence cap' if one.NumVars*n > 2000 else ''}")
print("\nPH instead solves n separate problems of ~{} vars each.".format(one.NumVars))

### Is there anything to hedge?

If every scenario independently wants the same year-1 decision, nonanticipativity is not
binding and the stochastic machinery is wasted effort. Always check this first.

In [ ]:
for nm, p, D in sc:
    m = build(d, invest_years=iy, demand=D); m.optimize()
    print(f"{nm}: year-1 builds ->",
          sorted(k[0] for k in m._y if k[1] == 1 and m._y[k].X > 0.5) or "none")

Genuine disagreement — low growth builds nothing, high growth builds several units. The
year-1 decision has to be made *before* knowing which world you're in.

## 4. The three strategies

This is the core of the notebook. Each strategy produces a **stage-1 decision**; we then fix
that decision and re-optimise stage 2 **separately in every scenario**. That gives the whole
cost *distribution*, not just the expectation — which is where the argument actually lives.

$$\text{EVPI} = \text{RP} - \text{WS}\ \ (\text{value of foresight}) \qquad
\text{VSS} = \text{EEV} - \text{RP}\ \ (\text{value of modelling uncertainty})$$

Theory requires $\text{WS} \le \text{RP} \le \text{EEV}$.


### Getting the three cases right — a trap worth naming

All three numbers must be **evaluated expectations** produced by identical machinery:

| Case | How it is computed |
|---|---|
| **PI** (WS) | Solve each scenario deterministically, take the probability-weighted average |
| **SP** (RP) | Take stage-1 from the extensive form, **fix it**, re-optimise stage 2 in each scenario, weight |
| **EV** (EEV) | Take stage-1 from the **mean-demand** problem, **fix it**, re-optimise stage 2 in each scenario, weight |

The trap: reading RP straight off `ef.ObjVal` while EEV comes from the evaluation path. Those
are **not the same measurement**. A MILP terminated at a 0.5% gap reports an objective above
the true optimum, whereas the evaluation path re-solves stage 2 to optimality for a fixed
stage-1 — so it is systematically tighter. Mix the two and you can produce a **negative VSS**,
which is theoretically impossible ($\text{EEV} \ge \text{RP}$ always, since the EV stage-1
decision is feasible for the stochastic problem but generally not optimal for it).

`three_case_comparison()` routes everything through one path. `RP` evaluated must equal
`ef.ObjVal` — that equality is the assertion that the plumbing is right.


In [ ]:
R = three_case_comparison(d, sc, iy, s1, mipgap=1e-6)
WS, RP, EEV = R['WS'], R['RP'], R['EEV']
ev_fix, sp_fix = R['ev_fix'], R['sp_fix']

print(f"extensive form solve value  {R['ef_obj']:10.2f}")
print(f"RP via evaluation path      {RP:10.2f}")
print(f"consistency check (must be ~0)  {RP - R['ef_obj']:+.6f}\n")
print(f"WS  (PI, perfect information)  {WS:10.2f}")
print(f"RP  (SP, stochastic optimum)   {RP:10.2f}")
print(f"EEV (EV strategy evaluated)    {EEV:10.2f}\n")
assert WS <= RP + 1e-6, "WS <= RP violated"
assert RP <= EEV + 1e-6, "RP <= EEV violated"
print("ordering WS <= RP <= EEV : verified")
print(f"\nEVPI = RP - WS  = {RP-WS:8.2f}   ({100*(RP-WS)/RP:.2f}% of RP)")
print(f"VSS  = EEV - RP = {EEV-RP:8.2f}   ({100*(EEV-RP)/RP:.2f}% of RP)")

In [ ]:
# Per scenario, PI must be <= BOTH other strategies -- it optimises each world freely.
import pandas as pd
rows = []
for strat in ['PI', 'SP', 'EV']:
    for r in R['per'][strat]:
        rows.append(dict(strategy=strat, **r))
df = pd.DataFrame(rows)
print("COST by scenario");  print(df.pivot(index='strategy', columns='scenario',
                                           values='cost').round(1).reindex(['PI','SP','EV']))
print("\nUNMET DEMAND by scenario"); print(df.pivot(index='strategy', columns='scenario',
                                           values='unmet').round(1).reindex(['PI','SP','EV']))
piv = df.pivot(index='scenario', columns='strategy', values='cost')
print("\nPI <= SP in every scenario:", bool((piv.PI <= piv.SP + 1e-6).all()))
print("PI <= EV in every scenario:", bool((piv.PI <= piv.EV + 1e-6).all()))

### Visual: scenarios and the two strategies

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':12,'axes.grid':True,'grid.alpha':0.3})
fig, ax = plt.subplots(1,2,figsize=(12,4.5))
for (nm,p,D),c in zip(sc,['#2471a3','#d68910','#c0392b']):
    ax[0].plot(d.years,[D['R2',t] for t in d.years],'o-',lw=2.8,color=c,
               label=f"R2 {float(nm.split('_g')[1])*100:.0f}%/yr (p={p:.2f})")
ax[0].plot(d.years,[d.D['R1',t] for t in d.years],'--',lw=2.5,color='grey',label='R1 (known)')
ax[0].set_xlabel('year'); ax[0].set_ylabel('demand'); ax[0].legend(fontsize=10)
ax[0].set_title('The uncertainty: Region 2 growth')
nev=sum(1 for k in ev_fix if ev_fix[k]>0.5); nsp=sum(1 for k in sp_fix if sp_fix[k]>0.5)
b=ax[1].bar(['EV\n(mean demand)','SP\n(stochastic)'],[nev,nsp],
            color=['#d68910','#196f3d'],width=0.55)
for bb,n,fx in zip(b,[nev,nsp],[ev_fix,sp_fix]):
    ax[1].text(bb.get_x()+bb.get_width()/2,n+0.12,
               f"{n} units\n{', '.join(sorted(k[0] for k in fx if fx[k]>0.5))}",
               ha='center',fontsize=10.5)
ax[1].set_ylim(0,nsp+1.8); ax[1].set_ylabel('year-1 units built')
ax[1].set_title('The here-and-now decision: SP hedges')
plt.tight_layout(); plt.show()

In [ ]:
# FIG -- EV under-builds, SP hedges
labels = ['low 1%', 'mid 7%', 'high 14%']
x = np.arange(3); w = .26
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for j, (st, c) in enumerate([('PI', CB['grey']), ('SP', CB['green']), ('EV', CB['red'])]):
    sub = df[df.strategy == st]
    ax[0].bar(x+(j-1)*w, list(sub.cost), w, color=c, label=st, edgecolor='k', lw=.4)
    ax[1].bar(x+(j-1)*w, list(sub.unmet), w, color=c, label=st, edgecolor='k', lw=.4)
ax[0].set(xticks=x, xticklabels=labels, ylabel='discounted cost', title='Cost by scenario')
ax[0].set_ylim(bottom=20000)
ax[0].annotate('EV blows up\nwhen growth is high', xy=(2+w, 55500), xytext=(0.30, 52500),
               fontsize=10.5, color=CB['red'],
               arrowprops=dict(arrowstyle='->', color=CB['red'], lw=1.4))
ax[1].set(xticks=x, xticklabels=labels, ylabel='unmet demand (units)',
          title='Unmet demand by scenario')
ax[0].legend(fontsize=10); ax[1].legend(fontsize=10)
plt.suptitle('EV under-builds; SP hedges. The mean hides this.', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# FIG -- what the hedge costs and what it buys
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
vals = [WS, RP, EEV]
ax[0].bar(['WS\n(PI)', 'RP\n(SP)', 'EEV\n(EV)'], vals,
          color=[CB['grey'], CB['green'], CB['red']], edgecolor='k', lw=.5)
for i, v in enumerate(vals):
    ax[0].text(i, v+120, f'{v:,.0f}', ha='center', fontweight='bold', fontsize=11)
ax[0].set_ylim(min(vals)*0.95, max(vals)*1.035)
ax[0].annotate('', xy=(0, RP), xytext=(1, RP),
               arrowprops=dict(arrowstyle='<->', color=CB['blue'], lw=1.8))
ax[0].text(0.5, RP+260, f'EVPI\n{RP-WS:,.0f}  ({100*(RP-WS)/RP:.1f}%)',
           ha='center', color=CB['blue'], fontweight='bold', fontsize=10)
ax[0].annotate('', xy=(1, EEV), xytext=(2, EEV),
               arrowprops=dict(arrowstyle='<->', color=CB['purple'], lw=1.8))
ax[0].text(1.5, EEV+260, f'VSS\n{EEV-RP:,.0f}  ({100*(EEV-RP)/RP:.1f}%)',
           ha='center', color=CB['purple'], fontweight='bold', fontsize=10)
ax[0].set(ylabel='expected cost', title='WS <= RP <= EEV')
sp_c = list(df[df.strategy == 'SP'].cost); ev_c = list(df[df.strategy == 'EV'].cost)
diff = [sp_c[i]-ev_c[i] for i in range(3)]
ax[1].bar(labels, diff, color=[CB['orange'] if v > 0 else CB['green'] for v in diff],
          edgecolor='k', lw=.5)
ax[1].axhline(0, color='k', lw=1)
for i, v in enumerate(diff):
    ax[1].text(i, v + (300 if v > 0 else -900), f'{v:+,.0f}', ha='center',
               fontweight='bold', fontsize=10.5)
ax[1].set(ylabel='SP cost - EV cost',
          title='The insurance trade\n(orange = premium paid, green = disaster avoided)')
plt.suptitle('What the hedge costs and what it buys', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt
piv = df.pivot(index='scenario', columns='strategy', values='cost')[['PI', 'SP', 'EV']]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
piv.plot(kind='bar', ax=ax[0], rot=0)
ax[0].set_title('Cost by scenario'); ax[0].set_ylabel('discounted cost')
ax[0].set_ylim(bottom=piv.min().min()*0.9)
un = df.pivot(index='scenario', columns='strategy', values='unmet')[['PI', 'SP', 'EV']]
un.plot(kind='bar', ax=ax[1], rot=0)
ax[1].set_title('Unmet demand by scenario'); ax[1].set_ylabel('units')
plt.tight_layout(); plt.show()

### Reading the result

This is the whole case for stochastic programming, and it is **not** visible in the
expected-cost column alone.

The **EV strategy under-builds.** It plans for average demand, so it commits to a small
year-1 program. In the low-growth scenario that's the *cheapest* of the three strategies —
it correctly avoided spending. But in the high-growth scenario it is catastrophic: lead
times mean it cannot react in time, so it eats a large unmet-demand penalty. Look at the
`unmet` row — EV's shortfall in the high scenario is an order of magnitude worse than SP's.

The **SP strategy hedges.** It builds more than EV in year 1, deliberately overpaying in the
world that turns out low, to buy the ability to serve the world that turns out high. Its
high-scenario cost sits close to PI's — it pre-positioned correctly for the tail. That
premium in the low scenario is the *insurance*, and VSS is what the insurance is worth.

**PI is not achievable** — it is a bound, and note it beats both other strategies in *every* scenario, not just on average. The gap from PI to SP (EVPI) is the value of
foresight you cannot buy. The gap from SP to EV (VSS) is the value of *modelling* the
uncertainty at all, and that one is entirely under your control.

This is the connection back to Part 1: EVPI is the same "value of foresight" quantity that
the rolling-horizon $W$ sweep was groping at, but measured against a real distribution
instead of a truncation parameter.

### Visual: EVPI and VSS

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,5.0))
b=ax.bar(['WS\n(perfect info)','RP\n(stochastic)','EEV\n(EV strategy)'],[WS,RP,EEV],
         color=['#7f8c8d','#196f3d','#d68910'],width=0.5)
for bb,v in zip(b,[WS,RP,EEV]):
    ax.text(bb.get_x()+bb.get_width()/2,v+120,f'{v:,.0f}',ha='center',fontsize=13,weight='bold')
ax.set_ylim(min(WS,RP,EEV)*0.94,max(WS,RP,EEV)*1.07); ax.axhline(RP,ls=':',color='#196f3d',lw=1.5)
ax.set_ylabel('expected discounted cost')
ax.annotate('',xy=(0.34,WS),xytext=(0.34,RP),arrowprops=dict(arrowstyle='<->',color='#c0392b',lw=2.4))
ax.text(0.42,(WS+RP)/2,f'EVPI = {RP-WS:,.0f} ({100*(RP-WS)/RP:.1f}%)\nvalue of foresight\n- cannot be bought',
        fontsize=11.5,color='#c0392b',va='center')
ax.annotate('',xy=(1.66,RP),xytext=(1.66,EEV),arrowprops=dict(arrowstyle='<->',color='#8e44ad',lw=2.4))
ax.text(1.58,(RP+EEV)/2,f'VSS = {EEV-RP:,.0f} ({100*(EEV-RP)/RP:.1f}%)\nvalue of modelling uncertainty\n- under your control',
        fontsize=11.5,color='#8e44ad',va='center',ha='right')
ax.set_title('WS $\\leq$ RP $\\leq$ EEV: what each gap is worth')
plt.tight_layout(); plt.show()

In [ ]:
# Where does the money go? Decompose SP's hedge against EV, scenario by scenario.
comp = df.pivot(index='scenario', columns='strategy', values='cost')
comp['SP_minus_EV'] = (comp['SP'] - comp['EV']).round(0)
comp['SP_minus_PI'] = (comp['SP'] - comp['PI']).round(0)
print(comp[['SP_minus_EV', 'SP_minus_PI']])
print("\nNegative SP_minus_EV = SP is CHEAPER than EV in that scenario.")
print("SP pays a premium where growth is low to avoid disaster where growth is high.")

## 5. Why decomposition is needed

The extensive form replicates the whole model per scenario. At three scenarios that's fine;
at hundreds it is not.

In [ ]:
ef = extensive_form(d, sc, iy, s1); ef.update()
one = build(d, invest_years=iy); one.update()
print(f"single deterministic model : {one.NumVars:5d} vars, {one.NumBinVars:4d} bins")
print(f"extensive form, {len(sc)} scen    : {ef.NumVars:5d} vars, {ef.NumBinVars:4d} bins")
print(f"\nprojected at 100 scenarios : ~{one.NumVars*100:6d} vars"
      f"  (pip licence cap is 2000)")
ef.optimize()
print(f"\nEF objective {ef.ObjVal:.1f}  (matches RP above)")

## 6. Progressive Hedging

**PH** (Rockafellar & Wets 1991) relaxes nonanticipativity, solves each scenario *as if
clairvoyant*, and iteratively forces agreement. Three steps per iteration:

1. Solve each scenario separately with a quadratic perturbation
2. Average scenario solutions at each non-leaf node — projection onto $\mathcal{N}$
3. Update Lagrange multipliers

$$x_i^{k+1} = \arg\min_{x_i}\Big\{h_i(x_i) + (w_i^k)^\top x_i + \tfrac{\rho}{2}\|x_i - z^k\|^2\Big\}$$
$$z^{k+1} = \sum_i \pi_i x_i^{k+1}, \qquad w_i^{k+1} = w_i^k + \rho\,(x_i^{k+1} - z^{k+1})$$

PH is ADMM in a scenario-product space under a probability-weighted inner product.

**What it buys you:** PH decomposes the **scenario** dimension, not the **time** dimension.
On a purely deterministic model it gives you nothing. What it does is make going stochastic
*affordable* — your existing deterministic model becomes the subproblem, and $n$ scenarios
become $n$ parallel solves of roughly that size instead of one $n\times$ extensive form.

### The binary trick that keeps subproblems MILPs

The quadratic penalty would normally make each subproblem a MIQP. But stage-1 decisions are
**binary**, and for $x\in\{0,1\}$ we have $x^2 = x$, so

$$\tfrac{\rho}{2}\|x-z\|^2 \;=\; \tfrac{\rho}{2}\big[x(1-2z) + z^2\big]$$

which is **linear**. Subproblems stay MILPs. This is what `ph()` implements.

In [ ]:
rows = []
for rho in [50, 100, 300, 900, 2500, 7000]:
    t0 = time.time()
    r = ph(d, sc, iy, s1, rho=rho, iters=60)
    val = evaluate_stage1(d, sc, iy, s1, r['z'])
    rows.append(dict(rho=rho, converged=r['resid'][-1] < 1e-4, iters=r['iters'],
                     final_resid=round(r['resid'][-1], 4),
                     PH_value=round(val, 1),
                     gap_vs_EF_pct=round(100*(val-RP)/RP, 2),
                     seconds=round(time.time()-t0, 1)))
pd.DataFrame(rows)

**Convergence is not monotone in $\rho$.** Small $\rho$ never forces agreement; large $\rho$
snaps to a poor point immediately; intermediate values cycle. There is a good region — one
of these recovers the extensive-form optimum to ~0.01% — but you cannot find it by
reasoning, only by sweeping.

This is the practical cost of PH on **mixed-integer** problems. PH's convergence proof
covers **convex** problems with a compact feasible set; on MILPs it is a *heuristic*, though
with adjustments it yields valid Lagrangian lower bounds (Gade et al. 2016; Boland et al.
2018). Never deploy PH on a MILP without a $\rho$ sweep and a termination rule that does not
assume convergence.

### Visual: PH residual by $\rho$

In [ ]:
import numpy as np
hist={}
for rho in [50,300,900,2500]:
    hist[rho]=ph(d,sc,iy,s1,rho=rho,iters=60)['resid']
fig,ax=plt.subplots(figsize=(9,4.6))
for rho,c in zip([50,300,900,2500],['#c0392b','#196f3d','#d68910','#2471a3']):
    ax.semilogy(range(1,len(hist[rho])+1),np.maximum(hist[rho],1e-6),lw=2.6,color=c,
                label=f'$\\rho$={rho}')
ax.axhline(1e-4,ls='--',color='k',lw=1.5); ax.text(30,1.5e-4,'tolerance',fontsize=11)
ax.set_xlabel('PH iteration'); ax.set_ylabel('primal residual (log)')
ax.legend(fontsize=11); ax.set_title('Convergence is NOT monotone in $\\rho$')
plt.tight_layout(); plt.show()

## 7. Block-asynchronous operation (APH)

Standard PH is **strongly synchronous**: every subproblem must finish before an iteration
completes. When subproblem runtimes vary — acute with integer variables, where subproblems
are MIPs — most processors idle waiting for the slowest.

**APH** (Eckstein, Watson & Woodruff, *Operations Research* 73(1), 2025) relaxes this two
ways: only a *subset* of subproblems need be solved per iteration ("block asynchronous"),
and results may be incorporated a bounded number of iterations after dispatch. It's derived
from **projective operator splitting** rather than Douglas–Rachford, which buys
**deterministic** convergence under a *fairness* condition on subproblem selection — rather
than the *almost sure* convergence of randomized alternatives — and needs no delay-based
under-relaxation.

The fairness condition (their Assumption A3) requires every index be selected at least once
every $M$ iterations. `ph()` enforces it with a deterministic round-robin. **Random sampling
would not guarantee it** — and if you skip it, the residual can hit zero spuriously because
stale subproblem solutions agree with a stale average.

The metric that matters is **total subproblem solves**, not iterations — in a parallel
deployment that's the wall-clock driver.

In [ ]:
rows = []
for bf in [1.0, 0.67, 0.34]:
    t0 = time.time()
    r = ph(d, sc, iy, s1, rho=300, iters=80, block_frac=bf)
    val = evaluate_stage1(d, sc, iy, s1, r['z'])
    rows.append(dict(block_frac=bf, scen_per_iter=max(1, round(bf*len(sc))),
                     iters=r['iters'], SUBPROBLEM_SOLVES=r['subsolves'],
                     converged=r['resid'][-1] < 1e-4,
                     gap_vs_EF_pct=round(100*(val-RP)/RP, 2),
                     seconds=round(time.time()-t0, 1)))
pd.DataFrame(rows)

In [ ]:
# FIG -- PH rho sensitivity + block asynchrony
hist = {}
for rho_ in [50, 300, 900, 2500, 7000]:
    rr = ph(d, sc, iy, s1, rho=rho_, iters=60)
    vv = evaluate_stage1(d, sc, iy, s1, rr['z'])
    hist[rho_] = (rr['resid'], rr['resid'][-1] < 1e-4, 100*(vv-RP)/RP)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for (rho_, (resid_, conv, gpct)), c in zip(hist.items(),
        [CB['grey'], CB['green'], CB['red'], CB['blue'], CB['orange']]):
    ax[0].plot(range(1, len(resid_)+1), np.maximum(resid_, 1e-6), lw=2, color=c,
               label=f"rho={rho_} {'converges' if conv else 'CYCLES'} ({gpct:+.2f}%)")
ax[0].set_yscale('log')
ax[0].set(xlabel='PH iteration', ylabel='primal residual (log)',
          title='Convergence is NOT monotone in rho')
ax[0].legend(fontsize=8.5, loc='center left')
lbl = ['3 of 3\nscenarios', '2 of 3\nscenarios', '1 of 3\nscenarios']
solves, gaps2 = [], []
for bf in [1.0, 0.67, 0.34]:
    rr = ph(d, sc, iy, s1, rho=300, iters=80, block_frac=bf)
    vv = evaluate_stage1(d, sc, iy, s1, rr['z'])
    solves.append(rr['subsolves']); gaps2.append(100*(vv-RP)/RP)
ax[1].bar(lbl, solves, color=CB['purple'], edgecolor='k', lw=.5)
for i, (sv, g2) in enumerate(zip(solves, gaps2)):
    ax[1].text(i, sv*0.5, f'{sv}\nsolves', ha='center', color='w',
               fontweight='bold', fontsize=11)
    ax[1].text(i, sv+4, f'{g2:+.2f}%', ha='center', fontsize=10)
ax[1].set(ylabel='total subproblem solves',
          title='Block-asynchronous (APH):\nfewer solves, same answer')
plt.suptitle('Progressive Hedging: rho sensitivity and block asynchrony',
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

Solving a subset per iteration cuts total subproblem solves substantially at comparable
solution quality. With three scenarios on one core this is a toy demonstration — the paper
reports the margin **widening** as the ratio of scenarios to cores grows, tested at
20,000–1,000,000 scenarios on 48–6,000 cores.

Note what block-asynchrony does **not** fix: it improves the coordination loop's efficiency.
It is not a remedy for the $\rho$ sensitivity in §6.

### Visual: subproblem solves

In [ ]:
dfG = pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(8,4.4))
b=ax.bar([f"{int(r.scen_per_iter)}/{len(sc)}\nscenarios" for _,r in dfG.iterrows()],
         dfG.SUBPROBLEM_SOLVES,color=['#7f8c8d','#2471a3','#196f3d'],width=0.55)
for bb,(_,r) in zip(b,dfG.iterrows()):
    ax.text(bb.get_x()+bb.get_width()/2,r.SUBPROBLEM_SOLVES+3,
            f"{int(r.SUBPROBLEM_SOLVES)} solves\ngap {r.gap_vs_EF_pct:+.2f}%",ha='center',fontsize=11)
ax.set_ylim(0,dfG.SUBPROBLEM_SOLVES.max()*1.3)
ax.set_ylabel('total subproblem solves')
ax.set_title('Block-asynchronous (APH): fewer solves, same answer')
plt.tight_layout(); plt.show()


### When is stochastic programming actually worth it?

EVPI and VSS answer different questions, and **a large EVPI does not imply stochastic
programming will help you.** Sweep the recourse asymmetry and watch them decouple.


In [ ]:
sweep = []
for pen in [30, 45, 90, 160, 250]:
    dd = Data(T=12); dd.slack_pen = pen
    scn = scenarios(dd)
    Rk = three_case_comparison(dd, scn, iy, s1, mipgap=1e-6)   # consistent evaluation
    ws, rp, eev = Rk['WS'], Rk['RP'], Rk['EEV']
    assert ws <= rp + 1e-6 and rp <= eev + 1e-6, f"ordering violated at pen={pen}"
    sweep.append(dict(slack_pen=pen,
                      EVPI_pct=round(100*(rp-ws)/rp, 2),
                      VSS_pct=round(100*(eev-rp)/rp, 2),
                      EV_builds=sum(1 for k in Rk['ev_fix'] if Rk['ev_fix'][k] > 0.5),
                      SP_builds=sum(1 for k in Rk['sp_fix'] if Rk['sp_fix'][k] > 0.5)))
dfS = pd.DataFrame(sweep); dfS

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(dfS.slack_pen, dfS.EVPI_pct, 'o-', lw=2.8, ms=10, color='#c0392b', label='EVPI %')
ax.plot(dfS.slack_pen, dfS.VSS_pct, 's-', lw=2.8, ms=10, color='#8e44ad', label='VSS %')
ax.axhline(0, ls=':', color='k', lw=1.5)
for _, r in dfS.iterrows():
    if r.EV_builds != r.SP_builds:
        ax.axvspan(r.slack_pen-9, r.slack_pen+9, color='#8e44ad', alpha=0.10)
ax.set_xlabel('unmet-demand penalty (recourse asymmetry)')
ax.set_ylabel('% of RP'); ax.legend(fontsize=12)
ax.set_title('EVPI rises monotonically. VSS does not.')
ax.annotate('EVPI large, VSS ~ 0:\nthe deterministic model\nalready gets it right',
            xy=(90, 0.2), xytext=(60, 5.2), fontsize=11, color='#8e44ad',
            arrowprops=dict(arrowstyle='->', color='#8e44ad'))
plt.tight_layout(); plt.show()


**Read the `EV_builds` vs `SP_builds` columns — that is the whole story.**

VSS is nonzero **exactly when the two strategies choose differently**, and is **exactly zero
otherwise**:

- **pen = 45 and 90** — both build 2 units, so VSS = 0.00 to numerical precision. At pen = 90,
  **EVPI is 4.9%** — foresight is worth a great deal — and yet the mean-value model already
  produces the optimal here-and-now decision. All the stochastic machinery buys you *nothing*.
- **pen = 160, 250** — SP builds 5 against EV's 2. It hedges **upward**, and VSS jumps to
  3.1% then 9.0%.
- **pen = 30** — SP builds 0 against EV's 2. Shortfall is now cheap, so SP hedges
  **downward**: don't sink capital you may not need. VSS is positive again, for the opposite
  reason.

Those exact zeros are the methodology working. An earlier version of this notebook read RP
off `ef.ObjVal` while EEV came from the evaluation path, and produced **VSS = −0.06%** at
pen = 90 — impossible, and easy to rationalise as rounding. It was not rounding: the
extensive form had stopped 22.8 above its true optimum inside a 0.5% gap, while the
evaluation path re-solved stage 2 exactly. **Two different measurements, differenced.** Hence
the assertions in the sweep loop — if `WS <= RP <= EEV` ever fails, the plumbing is wrong,
not the economics.

So the diagnostic is not "is EVPI big?" — it is **"does the distribution change my first
move?"** Run the EV problem and the SP problem, compare the stage-1 decisions, and if they
match, publish the deterministic model and say so. That check costs one extra solve and can
save you a formulation you don't need.



### Why is VSS *exactly* zero in some runs?

Not a bug, and not rounding. **VSS is exactly zero if and only if the EV problem and the SP
problem choose the same stage-1 decision.** When they do, both EEV and RP fix that identical
year-1 decision and re-optimise stage 2 through identical code — so they return identical
numbers and the difference is exactly 0.

That is the honest economic reading: over that range of penalties, the point forecast already
picks the right first move, so modelling the distribution changes nothing you would *do*.

But be aware **exact zeros are more common in this toy than they would be in a real model**,
for a structural reason: the stage-1 decision is a handful of **lumpy binaries** — build 0, 1,
2 or 3 units of 90–110 capacity at each site. The distribution only changes your answer if it
pushes you across an integer threshold. Over a wide band of penalties, both problems land on
the same integer.

Refine the decision space and the zeros become small positives:


In [ ]:
base = {'M1':110,'M2':110,'P1':100,'P2':100,'F1':90,'F2':90}
out = []
for div, mb, lab in [(1, 3, 'coarse (default)'), (3, 9, '3x finer units')]:
    dd = Data(T=10); dd.slack_pen = 90
    dd.cap_unit = {k: v/div for k, v in base.items()}; dd.max_builds = mb
    scn = scenarios(dd)
    ef_ = extensive_form(dd, scn, iy, s1, mipgap=1e-6); ef_.update()
    if ef_.NumVars > 1990:
        print(f'{lab}: {ef_.NumVars} vars over cap, skipped'); continue
    Rk = three_case_comparison(dd, scn, iy, s1, mipgap=1e-6)
    out.append(dict(granularity=lab, cap_unit=list(dd.cap_unit.values())[0],
                    VSS_pct=round(100*(Rk['EEV']-Rk['RP'])/Rk['RP'], 3),
                    EV_units=sum(1 for k in Rk['ev_fix'] if Rk['ev_fix'][k] > 0.5),
                    SP_units=sum(1 for k in Rk['sp_fix'] if Rk['sp_fix'][k] > 0.5)))
pd.DataFrame(out)


At 3× finer capacity units the two strategies diverge (SP picks a different count) and VSS
becomes **0.37%** rather than exactly 0. The absolute costs are *not* comparable across rows —
the non-learning site adder is charged per unit built, so more smaller units cost more — but
the VSS column is.

**Takeaway:** a zero VSS is a real answer ("your point forecast is adequate"), but interpret
it alongside how coarse your first-stage decision space is. With continuous or finely-grained
capacity you should expect small positive VSS rather than exact zeros.



## 8. Removing both limitations

Two things constrained everything above, and both are fixable:

1. **Lumpy, near-identical capacity units** (90–110) meant EV and SP often landed on the same
   integer, producing exact-zero VSS. Spreading the unit sizes to **80–120** makes the
   attainable capacity grid much finer *without* inflating cost — unlike shrinking the units,
   which multiplies the per-unit site adder.
2. **Only 3 scenarios**, because the extensive form replicates the model per scenario and hits
   the licence cap at $n=7$. **PH does not have this problem** — each subproblem is one
   scenario, ~280 variables, regardless of $n$.

`ph_three_case()` gets the SP decision from PH instead of the extensive form. WS and EEV were
never size-constrained (both are per-scenario solves), so all three cases scale.


In [ ]:
CU = {'M1':120, 'M2':95, 'P1':110, 'P2':85, 'F1':100, 'F2':80}   # spread 80-120
scale = []
for n in [3, 5, 7, 12, 20]:
    dd = Data(T=10); dd.slack_pen = 160; dd.cap_unit = dict(CU)
    scn = scenarios_n(dd, n)
    probe = build(dd, invest_years=iy); probe.update()
    ef_vars = probe.NumVars * n
    rp_ef = None
    if ef_vars <= 1990:                      # extensive form still fits
        e = extensive_form(dd, scn, iy, s1, mipgap=1e-6); e.optimize(); rp_ef = e.ObjVal
    Rk = ph_three_case(dd, scn, iy, s1, rho=300, iters=60)
    scale.append(dict(n_scen=n, EF_vars=ef_vars, EF_fits=ef_vars <= 1990,
                      RP_extensive=None if rp_ef is None else round(rp_ef, 1),
                      RP_via_PH=round(Rk['RP'], 1),
                      EVPI_pct=round(100*(Rk['RP']-Rk['WS'])/Rk['RP'], 2),
                      VSS_pct=round(100*(Rk['EEV']-Rk['RP'])/Rk['RP'], 2),
                      PH_subsolves=Rk['ph']['subsolves']))
dfSc = pd.DataFrame(scale); dfSc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].plot(dfSc.n_scen, dfSc.EF_vars, 'o-', lw=2.8, ms=10, color='#c0392b',
           label='extensive form')
ax[0].plot(dfSc.n_scen, [280]*len(dfSc), 's-', lw=2.8, ms=10, color='#196f3d',
           label='PH subproblem (each)')
ax[0].axhline(2000, ls='--', lw=2, color='k')
ax[0].text(3.2, 2140, 'pip licence cap', fontsize=11)
ax[0].set_xlabel('number of scenarios'); ax[0].set_ylabel('model size (variables)')
ax[0].set_title('Why PH scales'); ax[0].legend(fontsize=10.5, loc='upper left')
ax[1].plot(dfSc.n_scen, dfSc.EVPI_pct, 'o-', lw=2.9, ms=11, color='#c0392b', label='EVPI %')
ax[1].plot(dfSc.n_scen, dfSc.VSS_pct, 's-', lw=2.9, ms=11, color='#8e44ad', label='VSS %')
ax[1].axvline(7, ls=':', lw=2, color='grey')
ax[1].text(7.3, ax[1].get_ylim()[1]*0.9, 'EF cap', fontsize=11, color='grey', rotation=90, va='top')
ax[1].set_xlabel('number of scenarios'); ax[1].set_ylabel('% of RP')
ax[1].set_ylim(0, max(dfSc.EVPI_pct.max(), dfSc.VSS_pct.max())*1.2)
ax[1].set_title('Estimates shrink and settle as the tree refines'); ax[1].legend(fontsize=11.5)
plt.tight_layout(); plt.show()


Three results, and the third is the important one.

**PH validates against the extensive form where both are computable.** At $n=5$ and $n=7$
they agree to the digit. At $n=3$ PH is ~0.16% off — a reminder that on MILPs PH is a
heuristic, so validate at small $n$ before trusting it at large $n$.

**The exact zeros are gone.** With spread unit sizes VSS is nonzero at every scenario count.
The attainable capacity grid is now fine enough that the distribution genuinely shifts the
year-1 decision, which was the diagnosis in §4.

**Three scenarios overstates both metrics badly.** VSS falls from 7.31% at $n=3$ to 3.29% at
$n=20$ — a **factor of more than two** — and EVPI drifts from 7.10% to 5.29%. Both appear to
settle by $n \approx 12$–20. A coarse tree does not merely make these estimates noisy; it
**biases them upward**, because extreme scenarios carry too much probability mass and make the
future look more punishing than it is.

Practical consequence: **never quote VSS from a handful of scenarios as the justification for
a stochastic model.** Refine the tree until the metric stabilises, then quote it. Here that
means at least ~12 scenarios — which the extensive form cannot reach, and PH can.


## 9. How this relates to planning periods

The correspondence to Part 1 is direct:

| Rolling horizon (Part 1) | Stochastic program (Part 2) |
|---|---|
| Roll step $\Delta$ | Stage boundary in the scenario tree |
| Re-solve with new information | Scenario branching |
| Committed decisions in $[s, s+\Delta)$ | Stage-$t$ nonanticipative decisions |
| Ad hoc truncation at $W$ | Explicit distribution over futures |
| Gap to PF | EVPI |

**Rolling horizon is a poor man's stochastic program.** It captures "you don't know the
future" but never optimises against a *distribution* of futures — it commits to one path and
re-optimises when surprised. Nonanticipativity hedges across all branches at once.

Keep the categories straight: **PF / rolling / stochastic are formulations. PH is a solution
algorithm** for one of them. And the two axes are independent — everything in Part 1
(staggered investment periods, annualized capex, vintage efficiency, SOS2 learning) applies
unchanged inside each PH subproblem.

### Recommended sequence

1. Get the deterministic model right first (Part 1) — annualized capex, vintage efficiency,
   staggered periods. Bugs here corrupt everything downstream.
2. Check whether scenarios actually **disagree** on the here-and-now decision. If not, stop;
   you don't need this.
3. Build the extensive form at a small scenario count and verify $WS \le RP \le EEV$.
4. Report **VSS** — that's the argument for the added complexity. If VSS is near zero, the
   deterministic model was adequate and you should say so.
5. Only then scale up with PH, with a $\rho$ sweep and a bound from Gade et al. / Boland
   et al. rather than assumed convergence.

### Things to try
- More scenarios (`growths=` in `scenarios()`) — EVPI should grow, and the extensive form
  should degrade faster than PH
- `d.slack_pen = 45` — with symmetric recourse VSS collapses toward zero. Worth seeing:
  **asymmetric downside is what makes stochastic programming pay.**
- `stage1_years = [1, 5]` — a bigger nonanticipative block, harder for PH
- Lead time `d.lead = {s: 5 for s in d.sites}` — less reaction time, so EV should look worse